# 02. Stage-by-stage execution, with proof at every stage

Each section runs one stage through the CLI on a fresh scratch copy, then compares **only what that stage wrote** with
the golden MATLAB files. The reference counts come from the original `STAMPS.log`, read independently of Rustamps.

> **Scope.** These notebooks prove Stages 1-5 on `PATCH_1` of the golden dataset, starting from the raw
> `pscands.1.*` candidate files and ending at the Stage-5 patch products. Stages 6-8 are **not covered**:
> they need the merged four-patch golden data, which is not part of the local download.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "Cargo.toml").exists())
sys.path[:0] = [str(root), str(root / "notebooks")]

import matplotlib.pyplot as plt, numpy as np, pandas as pd
from IPython.display import display, Markdown
import rustamps_nb as nb
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_rows", 120)
print("binary:", nb.binary()); print("golden:", nb.require_golden())

In [ ]:
run = nb.stage_inputs("stagewise")
expected = nb.golden_log_counts()
proof, seconds = [], {}
def stage(n, files, counts):
    summary, seconds[n] = nb.run_stage(run, n)
    table = nb.compare_files(run, files)
    nb.must_pass(table, f"stage {n}")
    for claim, ok, evidence in counts:
        proof.append(nb.check(ok, claim, evidence))
    proof.append(nb.check(True, f"stage {n}: {len(table)} variables exact or within declared bound", f"{int((table.status=='EXACT').sum())} exact, {seconds[n]:.1f} s"))
    return table
read = lambda f, k: nb.read_mat(run / "PATCH_1" / f)[k]

## Stage 1. Load

Builds `ps1/ph1/bp1/la1/da1/hgt1` from the raw candidate files: projects lon/lat to local metres, rotates by the heading, sorts by azimuth, inserts the null master image.

In [ ]:
n_lines = sum(1 for _ in open(run / "PATCH_1" / "pscands.1.ij"))
t1 = stage(1, ["ps1.mat", "ph1.mat", "bp1.mat", "la1.mat", "da1.mat", "hgt1.mat"], [])
n_ps = int(read("ps1.mat", "n_ps").ravel()[0])
proof.append(nb.check(n_ps == n_lines == expected["stage1_candidates"], "stage 1 keeps all candidates", f"{n_ps} == {n_lines} == log"))
exact = t1[t1.variable.isin(["ij", "xy", "sort_ix", "ph", "bperp_mat", "lonlat"])]
proof.append(nb.check((exact.status == "EXACT").all(), "indices, coordinates, phase and baselines are bit-exact", ", ".join(exact.variable)))
display(t1[["file", "variable", "status", "n_diff", "max_abs", "bound", "pass"]])
xy = read("ps1.mat", "xy"); da = read("da1.mat", "D_A").ravel()
plt.figure(figsize=(6, 5)); plt.scatter(xy[:, 1], xy[:, 2], c=da, s=1, cmap="viridis"); plt.colorbar(label="D_A")
plt.xlabel("x (m)"); plt.ylabel("y (m)"); plt.title("Stage 1 candidates"); plt.show()

## Stage 2. Estimate gamma

Iterates a per-pixel topographic-error fit and an adaptive patch phase until the coherence stops changing; writes `pm1.mat`.

In [ ]:
t2 = stage(2, ["pm1.mat"], [])
g = nb.read_mat(nb.GOLDEN / "PATCH_1" / "pm1.mat")
proof.append(nb.check(float(read("pm1.mat", "n_trial_wraps").ravel()[0]) == float(g["n_trial_wraps"].ravel()[0]) and
                      int(read("pm1.mat", "i_loop").ravel()[0]) == int(g["i_loop"].ravel()[0]),
                      "same n_trial_wraps and the same number of iterations as StaMPS",
                      f"n_trial_wraps={float(g['n_trial_wraps'].ravel()[0]):.6f}, i_loop={int(g['i_loop'].ravel()[0])}"))
display(t2[~t2.variable.isin(["Nr", "coh_bins", "low_pass"])][["variable", "status", "n_diff", "max_abs", "bound", "pass"]])
plt.figure(figsize=(7, 4)); bins = np.linspace(0, 1, 60)
plt.hist(read("pm1.mat", "coh_ps").ravel(), bins, alpha=.6, label="Rustamps"); plt.hist(g["coh_ps"].ravel(), bins, histtype="step", color="k", label="StaMPS")
plt.xlabel("coherence"); plt.legend(); plt.title("Stage 2 coherence"); plt.show()

## Stage 3. Select

Chooses PS by a density-based coherence threshold per amplitude-dispersion bin and re-estimates their coherence; writes `select1.mat`.

In [ ]:
t3 = stage(3, ["select1.mat"], [])
ix, keep = read("select1.mat", "ix"), read("select1.mat", "keep_ix")
proof.append(nb.check(len(ix) == expected["stage3_initial"], "stage 3 initially selects the logged number of PS", f"{len(ix)}"))
proof.append(nb.check(int(keep.sum()) == expected["stage3_final"], "stage 3 keeps the logged number after re-estimation", f"{int(keep.sum())}"))
display(t3[["variable", "status", "n_diff", "max_abs", "bound", "pass"]])
gc = nb.read_mat(nb.GOLDEN / "PATCH_1" / "select1.mat")["coh_ps2"].ravel(); rc_ = read("select1.mat", "coh_ps2").ravel()
plt.figure(figsize=(5, 5)); plt.scatter(gc[::20], rc_[::20], s=2); plt.plot([0, 1], [0, 1], "r"); plt.xlabel("StaMPS"); plt.ylabel("Rustamps"); plt.title("coh_ps2"); plt.show()

## Stage 4. Weed

Drops adjacent duplicates and noisy arcs on a Delaunay network; writes `weed1.mat`.

In [ ]:
t4 = stage(4, ["weed1.mat"], [])
w1, w2 = read("weed1.mat", "ix_weed"), read("weed1.mat", "ix_weed2")
proof.append(nb.check(int(w1.sum()) == int(w2.sum()) == expected["stage4_kept"],
                      "both weed masks keep the logged number of PS (ps_weed.m sets ix_weed(ix_weed)=ix_weed2)", f"ix_weed={int(w1.sum())}, ix_weed2={int(w2.sum())}, log={expected['stage4_kept']}"))
display(t4[["variable", "status", "n_diff", "max_abs", "bound", "pass"]])

## Stage 5. Promote and merge

Promotes the kept PS to version 2 products and merges the patch list; writes `ps2/ph2/pm2/bp2/la2/hgt2/rc2`.

In [ ]:
t5 = stage(5, ["ps2.mat", "ph2.mat", "bp2.mat", "la2.mat", "hgt2.mat", "pm2.mat", "rc2.mat"], [])
n2 = int(read("ps2.mat", "n_ps").ravel()[0])
proof.append(nb.check(n2 == expected["stage4_kept"], "stage 5 promotes exactly the PS kept by stage 4", f"{n2}"))
display(t5[["file", "variable", "status", "n_diff", "max_abs", "bound", "pass"]])

## The check can fail: negative controls

A comparison that cannot fail proves nothing. Here we copy a golden file, change **one value in one array** by `1e-3`
(a hundred times below any plot-visible difference), and require the comparison to flag exactly that variable.

In [ ]:
import tempfile, shutil
from pathlib import Path
tmp = Path(tempfile.mkdtemp(dir=nb.WORK))
for name, var, delta in [("select1.mat", "ix", 1.0), ("ps2.mat", "xy", 1e-3)]:
    broken = nb.perturb_copy(nb.GOLDEN / "PATCH_1" / name, tmp / f"broken_{name}", var, delta)
    table = nb.evaluate(nb.compare_mat(broken, run / "PATCH_1" / name).assign(file=name))
    flagged = table[~table["pass"]].variable.tolist()
    proof.append(nb.check(flagged == [var], f"a one-element change in {name}:{var} is detected and nothing else is", f"flagged={flagged}"))
shutil.rmtree(tmp)

## Verdict

In [ ]:
verdict = pd.DataFrame(proof); display(verdict[["claim", "ok"]]); assert verdict.ok.all()
print("NOTEBOOK PASSED:", len(verdict), "checks;", {k: round(v, 1) for k, v in seconds.items()}, "seconds per stage")